# Multimodal Stimuli: Content Blocks and JSON-Authored Paths

A trial's `"stimulus"` doesn't have to be plain text — it can be a list of
[standard content blocks](https://docs.langchain.com/oss/python/langchain/messages)
(image/audio/file), so the same `ExpCard`/`ScannerModel` pipeline every
other tutorial uses also drives vision-language and audio-capable models
with zero changes outside the task card. This runs against the built-in
`mock-llm` family (no API key, no network) so this page executes for
real at build time; swap `model`/`family` for a real VLM provider to see
actual multimodal responses.

Two ways to build a block:

1. **`image_block()`/`audio_block()`/`file_block()`** — called from your
   own Python code, which already knows the path and is trusted to name
   any file on disk it wants.
2. **A JSON-authored `{"type": ..., "path": "..."}`** block, resolved
   automatically inside `gen_stimulus_prompt`. Since a task card can come
   from a third party (`download_lib()`, a hand-off from a collaborator),
   this path is restricted to a relative path with no `..` traversal —
   task-card JSON is untrusted input, so it can't be used to read
   arbitrary files off the host and ship their contents to a model
   provider.

In [1]:
import tempfile
from pathlib import Path

from psychscanner.datasets.prompts.multimodal import image_block

# A tiny real PNG, built with the Python API (path resolved by our own
# trusted code, no restriction on where it points).
media_dir = Path(tempfile.mkdtemp(prefix="psychscanner_multimodal_tutorial_"))
img_path = media_dir / "gradient.png"
img_path.write_bytes(
    bytes.fromhex(
        "89504e470d0a1a0a0000000d494844520000000200000002080600000072b60d24"
        "0000000c4944415478da6360606060000000050001a5f645400000000049454e44ae426082"
    )
)

block = image_block(img_path)
print(block["type"], block["mime_type"], len(block["base64"]), "base64 chars")

/Users/saurabhext/Documents/Projects/PSYCHSCANNER/psychscanner/.venv311/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


image image/png 96 base64 chars


## Running a multimodal trial end to end

A trial's `"stimulus"` is just this block plus a text block, wrapped in a
task card — same shape `ExpCard`/`ScannerModel` already expects.

In [2]:
from psychscanner import ExpCard, ExpCardInit, ScannerModel, to_csv

task_file = {
    "taskname": "multimodal_demo",
    "tasktype": "task",
    "instructions": {"definition": ["Describe what you see in the image."]},
    "context_present": False,
    "chain_type": "item",
    "parser": None,
    "items": {
        "img_1": [{
            "trcode": "img_1",
            "stimulus": [block, {"type": "text", "text": "What do you see?"}],
        }],
    },
}

proj_dir = Path(tempfile.mkdtemp(prefix="psychscanner_multimodal_tutorial_"))
card = ExpCardInit(
    model="mock-chat-model", family="mock-llm",
    task_file=task_file, cogtype="no", nsim=1, memory="SingleTurn",
    proj_dir=proj_dir,
)
scanner = ScannerModel(expcard=ExpCard(card))
results = scanner.run()

df = to_csv(scanner, path=proj_dir)
df.select(["trcode", "stimulus", "pred_resp_raw"])

----<PROJECT AND DATA ROOT DIRECTORY>----


	Project root dir: /var/folders/2f/1mng0xs503787mwn287f9_d00000gn/T/psychscanner_multimodal_tutorial_bwmt7rq7


	Simulation data root dir: /var/folders/2f/1mng0xs503787mwn287f9_d00000gn/T/psychscanner_multimodal_tutorial_bwmt7rq7/DEFAULTPROJ/multimodal_demo/mock-llm_mock-chat-model_SingleTurn


----<>----


--<chat model>-- metadata={'lc_versions': {'langchain-core': '1.5.3', 'langchain': '1.3.14'}} output_version=None model_name='mock-chat-model' repeat_buffer_length=10


...

2026-08-22 15:45:17.118 | INFO     | psychscanner.session_tunnel.session_tunnel:scan_checkpoint:184 - scan-checkpoint


----<scanned runs>---- i = 0


2026-08-22 15:45:17.120 | CRITICAL | psychscanner.session_tunnel.session_tunnel:end_checkpoint:163 - END


Saved 1 rows → /var/folders/2f/1mng0xs503787mwn287f9_d00000gn/T/psychscanner_multimodal_tutorial_bwmt7rq7/DEFAULTPROJ_multimodal_demo_mock-chat-model_SingleTurn_20260822_154517.csv


trcode,stimulus,pred_resp_raw
str,str,str
"""img_1""","""[{""type"": ""image"", ""mime_type""…","""[{'type': 'image', 'base64': '…"


## JSON-authored paths: the untrusted-input case

The same block shape, but written directly in task-card JSON with a
`"path"` field instead of a pre-built `"base64"` — no Python call needed.
`resolve_path_block()` handles this inside `gen_stimulus_prompt`
automatically. A **relative** path (resolved against the current working
directory) works exactly like the `image_block()` call above:

In [3]:
import os
from psychscanner.datasets.prompts.task_prompts import gen_stimulus_prompt

os.chdir(media_dir)  # so "gradient.png" resolves relative to cwd, not an absolute path

trstim = {
    "stimulus": [{"type": "image", "path": "gradient.png"}, {"type": "text", "text": "q"}],
    "trcode": "feat_1",
    "context_present": False,
}
msg = gen_stimulus_prompt(trstim)
print(msg.content[0]["type"], msg.content[0]["mime_type"], "path" in msg.content[0])

image image/png False


But an **absolute** path or a `../` escape is rejected — this is the
fix for a real finding from a code-review pass on this repo: without it,
a task card from an untrusted source (a hand-off, a compromised
third-party library fetch) could name a path like `/etc/passwd` or
`~/.aws/credentials` and have its contents silently base64-embedded into
the prompt sent to whatever LLM provider is configured.

In [4]:
for bad_path in ["/etc/passwd", "../../etc/passwd"]:
    try:
        gen_stimulus_prompt({
            "stimulus": [{"type": "file", "path": bad_path}],
            "trcode": "x", "context_present": False,
        })
        print(f"{bad_path!r}: NOT rejected (bug!)")
    except ValueError as exc:
        print(f"{bad_path!r}: rejected -- {exc}")

'/etc/passwd': rejected -- multimodal stimulus 'path' must be a relative path with no '..' components (task-card JSON is untrusted input) -- got '/etc/passwd'
'../../etc/passwd': rejected -- multimodal stimulus 'path' must be a relative path with no '..' components (task-card JSON is untrusted input) -- got '../../etc/passwd'


## Takeaways

1. Multimodal stimuli are just content-block lists in `"stimulus"` — the
   rest of the pipeline (memory, feedback, `to_csv()`) doesn't need to
   know or care.
2. `image_block()`/`audio_block()`/`file_block()` (trusted, your own
   Python code chooses the path) have no path restriction. JSON-authored
   `{"path": ...}` blocks (untrusted, task-card content) are restricted to
   a relative path with no `..` traversal.
3. `externalize_media()` (used internally by `ScannerModel` before
   writing checkpoints) content-addresses these blobs by `sha256(bytes)`
   on disk rather than inlining base64 into every `.psyscan` file — see
   `src/psychscanner/scanner_models/media_store.py` if you're inspecting
   checkpoint files directly rather than going through `to_csv()`.

## Further reading

1. **[LangChain multimodal content blocks](https://docs.langchain.com/oss/python/langchain/messages)**
   — the standard content-block format `image_block`/`audio_block`/
   `file_block`/`resolve_path_block` all build, shared across every
   LangChain-compatible provider.
2. **[OWASP Top 10: A01 Broken Access Control](https://owasp.org/Top10/A01_2021-Broken_Access_Control/)**
   — the general class of vulnerability `resolve_path_block()`'s
   containment check closes (path traversal / unrestricted file access
   from untrusted input).